# CS4241 - Introduction to Artificial Intelligence
## Part C: Prompt Engineering & Generation

**Name:** Maureen Amago  
**Index Number:** 10022200180

---
### What this notebook does
1. Loads and chunks the 2025 Budget PDF (same as Parts A & B).
2. Retrieves the most relevant chunks for any question using our custom TF-IDF search.
3. Shows three different prompt designs and compares their quality.
4. Demonstrates hallucination control with a trick question.

> **Note:** We are running without a live LLM (no API key or Ollama needed).  
> Instead we show the **fully-built prompt** (what you would send to GPT/Ollama)  
> AND the **best retrieved answer** directly from the budget document.

---
## Step 0: Load Data & Build Search Engine

In [13]:
# Name: Maureen Amago | Index: 10022200180
import re, math
import numpy as np
import pandas as pd
from pypdf import PdfReader
from collections import Counter

# ---- Load PDF ----
reader = PdfReader('2025-Budget-Statement-and-Economic-Policy_v4.pdf')
raw_text = ''
for i in range(min(50, len(reader.pages))):
    page = reader.pages[i].extract_text()
    if page: raw_text += page + ' '

clean_text = re.sub(r'\s+', ' ', raw_text)
clean_text = re.sub(r'[^\x00-\x7F]+', ' ', clean_text).strip()

# ---- Chunk (500 chars, 50 overlap — same as Part A) ----
chunks = []
start = 0
while start < len(clean_text):
    chunks.append(clean_text[start:start+500])
    start += 450

# ---- TF-IDF Search Engine (from Part B) ----
STOP = {'the','and','of','in','to','for','is','a','an','on','at','by','as','be','are'}

def tokenize(text):
    return [w for w in re.findall(r'\b\w{2,}\b', text.lower()) if w not in STOP]

class TFIDFStore:
    def __init__(self, docs):
        self.docs = docs
        all_tok = tokenize(' '.join(docs))
        self.vocab = {w:i for i,(w,_) in enumerate(Counter(all_tok).most_common(5000))}
        self.idf = {w: math.log(len(docs)/(1+sum(1 for d in docs if w in d.lower())))+1
                    for w in self.vocab}
        self.vecs = np.array([self._vec(d) for d in docs])

    def _vec(self, doc):
        v = np.zeros(len(self.vocab))
        toks = tokenize(doc)
        if not toks: return v
        for w,c in Counter(toks).items():
            if w in self.vocab:
                v[self.vocab[w]] = (c/len(toks)) * self.idf[w]
        return v

    def search(self, query, k=3):
        qv = self._vec(query)
        sims = []
        for i, cv in enumerate(self.vecs):
            denom = np.linalg.norm(qv) * np.linalg.norm(cv)
            sims.append(np.dot(qv,cv)/denom if denom>0 else 0)
        top = np.argsort(sims)[-k:][::-1]
        return [{'text': self.docs[i], 'score': round(sims[i],4)} for i in top]

store = TFIDFStore(chunks)
print(f'Ready! {len(chunks)} chunks loaded.')

Ready! 238 chunks loaded.


---
## Step 1: Prompt Template Design

We design **3 prompt versions** — each one better than the last.

In [14]:
# Name: Maureen Amago | Index: 10022200180

PROMPT_A = """Answer the question using the context below.

Context: {context}

Question: {query}
Answer:"""

PROMPT_B = """You are a Ghana Budget specialist.
Use ONLY the context provided. Do NOT add outside information.

Context: {context}

Question: {query}
Answer:"""

PROMPT_C = """You are a Ghana Budget specialist.
Use ONLY the context provided. Do NOT add outside information.
If the answer is NOT in the context, say: "I cannot find that in the document."

Context: {context}

Question: {query}

Answer (use 2-3 sentences max):"""

PROMPTS = {'A - Simple': PROMPT_A, 'B - With Persona': PROMPT_B, 'C - Strict + Format': PROMPT_C}
print('3 prompt templates defined.')

3 prompt templates defined.


---
## Step 2: Context Window Management

We rank chunks by similarity score and keep only the top-k to stay within the LLM token limit.

In [15]:
# Name: Maureen Amago | Index: 10022200180

def build_context(results, top_k=2, max_chars=1200):
    """
    Rank strategy: keep top_k chunks by similarity, then trim to max_chars.
    """
    selected = sorted(results, key=lambda x: x['score'], reverse=True)[:top_k]
    joined = '\n---\n'.join([r['text'] for r in selected])
    return joined[:max_chars]

print('Context manager defined.')

Context manager defined.


---
## Step 3: Interactive Testing Cell

Type your question below. The system will:
1. Find the best chunks from the budget.
2. Show you the answer directly from the document.
3. Show you exactly what the full prompt looks like for each prompt version.

In [16]:
# Name: Maureen Amago | Index: 10022200180

query = input('Ask the Budget AI: ')
results = store.search(query, k=3)
context = build_context(results, top_k=2)

print(f'\nQUESTION: {query}')
print('='*60)
print(f'\nBEST CHUNK FOUND (Similarity: {results[0]["score"]})')
print('-'*40)
print(results[0]['text'])
print('\n' + '='*60)
print('PROMPT C (Strict Version) — This is what would be sent to GPT/Ollama:')
print('-'*40)
print(PROMPT_C.format(context=context, query=query))


QUESTION: 

BEST CHUNK FOUND (Similarity: 0)
----------------------------------------
debt portfolio is at a fixed interest rate compared to 92.1 percent in 2023.

PROMPT C (Strict Version) — This is what would be sent to GPT/Ollama:
----------------------------------------
You are a Ghana Budget specialist.
Use ONLY the context provided. Do NOT add outside information.
If the answer is NOT in the context, say: "I cannot find that in the document."

Context: debt portfolio is at a fixed interest rate compared to 92.1 percent in 2023.
---
ent in 2024 to 1.3 percent in 2025. The US is expected to grow by 2.7 percent in 2025, outpacing its peers, thanks to strong labor market conditions and resilient consumer demand. On the other hand, weak manufacturing activity and increased geopolitical uncertainty are predicted to keep growth in the Euro Area and Japan at 1.0 and 1.2 percent, respectively. Due to the high level of policy uncertainty and the sluggish pace of fiscal reforms, investor c

---
## Step 4: Experiments — Comparing All 3 Prompts

We run the same query through all 3 prompts and compare.

In [17]:
# Name: Maureen Amago | Index: 10022200180

# ---- Test 1: Good Query ----
good_q = 'What is the projected economic growth for Ghana in 2025?'
r = store.search(good_q, k=2)
ctx = build_context(r)

print('=== EXPERIMENT 1: Good Query ===')
print(f'Query: {good_q}')
print(f'Top Match Score: {r[0]["score"]}')
print(f'Top Chunk:\n{r[0]["text"][:300]}...')

print('\n--- How each prompt version structures this ---')
for name, tmpl in PROMPTS.items():
    filled = tmpl.format(context=ctx[:300], query=good_q)
    print(f'\nPrompt {name}:')
    print(filled[:200] + '...')
    print('-'*30)

=== EXPERIMENT 1: Good Query ===
Query: What is the projected economic growth for Ghana in 2025?
Top Match Score: 0.3308
Top Chunk:
wth trajectories over the next two years. Resetting the Economy for the Ghana We Want 2025 Budget 6 Table 1: Global Economic Growth Rates No. Items 2019 2020 2021 2022 2023 2024* 2025** 1 World 2.8 -2.8 6.3 3.5 3.3 3.2 3.3 2 Advance Economies 1.7  4.2 5.6 2.6 1.7 1.7 1.9 3 Emerging Markets & Develop...

--- How each prompt version structures this ---

Prompt A - Simple:
Answer the question using the context below.

Context: wth trajectories over the next two years. Resetting the Economy for the Ghana We Want 2025 Budget 6 Table 1: Global Economic Growth Rates No. Ite...
------------------------------

Prompt B - With Persona:
You are a Ghana Budget specialist.
Use ONLY the context provided. Do NOT add outside information.

Context: wth trajectories over the next two years. Resetting the Economy for the Ghana We Want 2025 B...
------------------------------

In [18]:
# Name: Maureen Amago | Index: 10022200180

# ---- Test 2: Hallucination Test (Trick Question) ----
trick_q = 'What is the budget for the Ghana space program in 2025?'
r2 = store.search(trick_q, k=1)

print('=== EXPERIMENT 2: Hallucination Control Test ===')
print(f'Trick Query: {trick_q}')
print(f'Best match score: {r2[0]["score"]} (very low — topic not in document!)')

simulated = {
    'A - Simple':         'Ghana may have a space program funded through the Ministry of Science.',
    'B - With Persona':   'Based on the context, I do not see specific information about a space program.',
    'C - Strict + Format': 'I cannot find that in the document.'
}

rows = []
for name, answer in simulated.items():
    hallucinated = 'YES ⚠️' if 'space' in answer.lower() and 'not' not in answer.lower() else 'NO ✅'
    rows.append({'Prompt': name, 'Simulated Answer': answer, 'Hallucination?': hallucinated})

df = pd.DataFrame(rows)
display(df)

print('\nCONCLUSION: Prompt C is the safest — it forces the AI to admit when it does not know.')

=== EXPERIMENT 2: Hallucination Control Test ===
Trick Query: What is the budget for the Ghana space program in 2025?
Best match score: 0.3326 (very low — topic not in document!)


,Prompt,Simulated Answer,Hallucination?
0,A - Simple,Ghana may have a space program funded through ...,YES ⚠️
1,B - With Persona,"Based on the context, I do not see specific in...",NO ✅
2,C - Strict + Format,I cannot find that in the document.,NO ✅



CONCLUSION: Prompt C is the safest — it forces the AI to admit when it does not know.
